# 2.10 · 贝叶斯估计 / Bayesian Estimation

> **课程定位**
> MLE 的"对手戏"：参数不再是常数而是**随机变量**，估计 = 算后验分布。本课用共轭先验做到**全程解析、零 MCMC**（MCMC 在 2.12 预告、Part 18 正篇）。三个高光：**Beta-Binomial 完整推导、"正则化 = MAP"、贝叶斯 A/B 测试**。
> Parameters become random variables; estimation becomes computing posteriors. Fully analytic via conjugacy — no MCMC needed yet.

> 💡 **面试相关**
> - "频率 vs 贝叶斯的区别" ★★★★
> - "可信区间 vs 置信区间" ★★★★（2.5 的坑在这里反转）
> - "L2 正则化的贝叶斯解释" ★★★★（ML 岗高频）
> - "冷启动/小样本怎么估计转化率" ★★★★（先验的工业价值）

---

## 目录
1. [范式对决：频率 vs 贝叶斯 ⭐](#1)
2. [Beta-Binomial 共轭：完整推导 ⭐](#2)
3. [先验 = 伪计数：直觉与冷启动](#3)
4. [序贯更新：今天的后验是明天的先验](#4)
5. [可信区间：终于可以说"95% 概率"了](#5)
6. [MAP 与 "正则化 = 先验" ⭐](#6)
7. [先验敏感性与数据淹没](#7)
8. [实战：贝叶斯 A/B 测试 ⭐](#8)
9. [小结](#9)


<a id="1"></a>
## 1. 范式对决 / The Two Paradigms

$$\underbrace{p(\theta \mid \mathcal{D})}_{\text{后验}} = \frac{\overbrace{p(\mathcal{D} \mid \theta)}^{\text{似然}} \; \overbrace{p(\theta)}^{\text{先验}}}{\underbrace{p(\mathcal{D})}_{\text{证据(归一化)}}} \;\;\propto\;\; \text{似然} \times \text{先验}$$

| | 频率学派 / Frequentist | 贝叶斯 / Bayesian |
|---|---|---|
| $\theta$ 是什么 | 固定未知常数 | **随机变量**（不确定性的表达）|
| 输出 | 点估计 + CI（程序质保）| **整个后验分布** |
| "95%" 的含义 | 程序长期覆盖率（2.5 的绕口令）| **"θ 有 95% 概率在区间内"**（字面成立！）|
| 小样本 | 不稳（MLE 偏差、CI 失真）| 先验兜底 ⭐ |
| 代价 | — | 必须选先验（主观性之争）|

**务实立场**：不站队。大样本两者趋同（数据淹没先验）；小样本/层级结构/序贯决策用贝叶斯更顺手；监管报告/标准检验用频率更通行。
Pragmatism: they converge at large n. Bayes shines at small n, hierarchies, and sequential decisions; frequentism rules regulatory reporting.


<a id="2"></a>
## 2. Beta-Binomial 共轭：完整推导 ⭐ / The Conjugate Pair

**任务**：估计转化率 $p$。观测 $k$ 成功 / $n$ 次试验。

**先验** Beta$(\alpha, \beta)$：$\quad p(\theta) \propto \theta^{\alpha-1}(1-\theta)^{\beta-1}$
**似然** Binomial：$\quad p(k \mid \theta) \propto \theta^{k}(1-\theta)^{n-k}$

**后验**（直接乘，指数相加）：
$$p(\theta \mid k) \;\propto\; \theta^{(\alpha + k) - 1}(1-\theta)^{(\beta + n - k) - 1} \;=\; \mathrm{Beta}(\alpha + k,\; \beta + n - k)$$

**共轭** = 后验和先验同族 → **更新只是加法**：$\alpha \mathrel{+}= k$，$\beta \mathrel{+}= n-k$。零积分零 MCMC。

后验的三个点估计：
$$\text{均值} = \frac{\alpha+k}{\alpha+\beta+n}, \qquad \text{众数(MAP)} = \frac{\alpha+k-1}{\alpha+\beta+n-2}, \qquad \text{MLE} = \frac{k}{n}$$


In [ ]:
import numpy as np
import scipy.stats as st
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)

# 可视化: 先验 → 数据 → 后验 / Prior x likelihood -> posterior
alpha0, beta0 = 2, 8                  # 先验: "转化率大概 20% 上下" / prior belief ~20%
k, n = 9, 20                          # 数据: 9/20 = 45% (!)

theta = np.linspace(0, 1, 400)
prior = st.beta(alpha0, beta0)
post = st.beta(alpha0 + k, beta0 + n - k)

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(theta, prior.pdf(theta), "C0--", lw=2, label=f"prior Beta({alpha0},{beta0}), mean={prior.mean():.2f}")
ax.plot(theta, st.binom.pmf(k, n, theta)/np.trapezoid(st.binom.pmf(k, n, theta), theta),
        "C2:", lw=2, label=f"likelihood (k={k}/n={n}), MLE={k/n:.2f}")
ax.plot(theta, post.pdf(theta), "C3-", lw=2.5, label=f"posterior Beta({alpha0+k},{beta0+n-k}), mean={post.mean():.2f}")
ax.legend(); ax.set_xlabel("θ (转化率)")
ax.set_title("Posterior = compromise between prior and data")
plt.tight_layout(); plt.show()

print(f"先验均值 {prior.mean():.3f}  | MLE {k/n:.3f}  | 后验均值 {post.mean():.3f} ← 在两者之间")


**后验落在先验和 MLE 之间**——精确公式是加权平均：
$$\mathbb{E}[\theta \mid \mathcal{D}] = \underbrace{\frac{\alpha+\beta}{\alpha+\beta+n}}_{\text{先验权重}} \cdot \frac{\alpha}{\alpha+\beta} \;+\; \underbrace{\frac{n}{\alpha+\beta+n}}_{\text{数据权重}} \cdot \frac{k}{n}$$
数据越多，天平越倒向 MLE。


<a id="3"></a>
## 3. 先验 = 伪计数：直觉与冷启动 / Priors as Pseudo-counts

Beta$(\alpha, \beta)$ 的最佳读法：**"我提前看过 $\alpha$ 次成功、$\beta$ 次失败"**。

| 先验 | 伪计数 | 语义 |
|---|---|---|
| Beta(1, 1) = Uniform | 各 1 | "毫无头绪"（Laplace 平滑的来源！）|
| Beta(2, 8) | 2 成 8 败 | "大概 20%，但不太确定" |
| Beta(20, 80) | 20 成 80 败 | "相当确定在 20% 附近" |

**工业冷启动场景** ⭐：新商品上架 3 次浏览 1 次购买——MLE 说转化率 33%（荒谬）。挂上类目历史先验 Beta(5, 95)：后验均值 = 6/103 ≈ 5.8%（合理）。**0.9 节朴素贝叶斯的 Laplace 平滑、推荐系统的"贝叶斯平均评分"全是这一招**。
New item: 1 sale in 3 views — MLE says 33%. With a category prior Beta(5,95), the posterior says 5.8%. Laplace smoothing and Bayesian average ratings are this exact move.


In [ ]:
# 冷启动排序: 贝叶斯平均 vs 原始比例 / Cold-start ranking
items = [("老品 A", 120, 2400), ("老品 B", 80, 2100), ("新品 C", 3, 9), ("新品 D", 0, 2)]
a0, b0 = 5, 95                      # 类目先验: ~5% 转化

print(f"{'item':<8} {'data':>10} {'MLE':>7} {'Bayes后验均值':>12}   排序变化")
for name, k, n in items:
    mle = k/n if n else float("nan")
    bayes = (a0 + k) / (a0 + b0 + n)
    print(f"{name:<8} {f'{k}/{n}':>10} {mle:>7.1%} {bayes:>11.1%}")
print("\nMLE 排序: C(33%) 荒谬地排第一; 贝叶斯排序: 老品凭真实数据在前, 新品回归先验")


<a id="4"></a>
## 4. 序贯更新 / Sequential Updating

共轭的第二重红利：**今天的后验 = 明天的先验**。数据流式到达时逐批更新，**最终后验与一次性全量计算完全相同**（似然连乘的交换律）。
Today's posterior is tomorrow's prior — and the result is order-independent, identical to batch processing.


In [ ]:
# 流式更新转化率 / Streaming updates
true_p = 0.12
a, b = 1, 1                                   # 从无知开始 / start ignorant
print(f"{'batch':>6} {'累计数据':>10} {'后验均值':>9} {'95%可信区间':>22}")
total_k = total_n = 0
for batch in range(1, 6):
    n_new = 100
    k_new = rng.binomial(n_new, true_p)
    a, b = a + k_new, b + (n_new - k_new)     # 整个"学习"就这一行 / learning = one line
    total_k += k_new; total_n += n_new
    post = st.beta(a, b)
    lo, hi = post.ppf([0.025, 0.975])
    print(f"{batch:>6} {f'{total_k}/{total_n}':>10} {post.mean():>9.3f} [{lo:.3f}, {hi:.3f}]")
print(f"\n真值 p = {true_p} — 区间稳步收窄并罩住真值")


<a id="5"></a>
## 5. 可信区间 / Credible Intervals

**95% 可信区间**：后验分布的中间 95%（`post.ppf([0.025, 0.975])`）。

**与 CI 的本质区别**（2.5 的坑在此反转）：
- CI："程序长期 95% 次数罩住真值"（θ 固定，区间随机）
- **可信区间："θ 有 95% 概率落在这个区间"**——大众直觉的那句话，**在贝叶斯框架里字面成立**（因为 θ 本来就是随机变量）

> 💡 面试加分句："Everyone interprets confidence intervals as credible intervals anyway — Bayes is the framework where that interpretation is actually licensed."


<a id="6"></a>
## 6. MAP 与"正则化 = 先验" ⭐ / MAP & Regularization

**MAP**（最大后验）= 后验的众数：
$$\hat\theta_{\mathrm{MAP}} = \arg\max_\theta \big[\log p(\mathcal{D}\mid\theta) + \log p(\theta)\big] = \text{MLE 目标} + \text{先验惩罚}$$

**对参数取高斯先验** $\theta_j \sim \mathcal{N}(0, \tau^2)$：
$$-\log p(\theta) = \frac{1}{2\tau^2}\|\theta\|_2^2 + \text{const} \;\;\Rightarrow\;\; \boxed{\text{L2 正则(Ridge)} = \text{高斯先验的 MAP}}$$

**取 Laplace 先验** $p(\theta_j) \propto e^{-|\theta_j|/b}$：
$$\boxed{\text{L1 正则(Lasso)} = \text{Laplace 先验的 MAP}}$$

接通三条线：
- 2.9 的"损失 = 噪声模型" + 本课"正则 = 先验" → **目标函数 = 完整的概率模型**
- λ 越大 = 先验越窄 = 越不信数据 → Part 4 的 Ridge/Lasso 有了统计学出生证明
- 2.9 的"完美分离 MLE 发散" → 加先验（正则）后 MAP 有限——贝叶斯救场实锤


In [ ]:
# 数值验证: Ridge 解 = 高斯先验 MAP / Ridge == Gaussian-prior MAP
import scipy.optimize as opt

n_, d = 50, 3
X = rng.normal(size=(n_, d))
w_true = np.array([2.0, -1.0, 0.5])
y = X @ w_true + rng.normal(0, 1, n_)
lam, tau2, sigma2 = 2.0, None, 1.0
tau2 = sigma2 / lam                            # λ = σ²/τ² 的换算

# Ridge 闭式解 / Ridge closed form
w_ridge = np.linalg.solve(X.T @ X + lam*np.eye(d), X.T @ y)

# MAP: -log posterior = ½σ⁻²‖y-Xw‖² + ½τ⁻²‖w‖²
nlp = lambda w: 0.5/sigma2*np.sum((y - X@w)**2) + 0.5/tau2*np.sum(w**2)
w_map = opt.minimize(nlp, np.zeros(d)).x

print(f"Ridge (λ={lam}):  {w_ridge.round(4)}")
print(f"MAP (τ²=σ²/λ):   {w_map.round(4)}   ← 完全相同")


<a id="7"></a>
## 7. 先验敏感性与数据淹没 / Prior Sensitivity & Data Swamping

对先验的标准质疑："结论不就是你先验的回声吗？" 回答：**画敏感性图**——


In [ ]:
# 三种迥异先验 × 三种数据量 / Three priors x three data sizes
priors = {"乐观 Beta(8,2)": (8,2), "无知 Beta(1,1)": (1,1), "悲观 Beta(2,8)": (2,8)}
true_p = 0.35
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
theta = np.linspace(0, 1, 300)
for ax, n_ in zip(axes, [10, 100, 2000]):
    k_ = rng.binomial(n_, true_p)
    for (name, (a0, b0)), c in zip(priors.items(), ["C0", "C2", "C3"]):
        ax.plot(theta, st.beta(a0+k_, b0+n_-k_).pdf(theta), c, lw=1.8, label=name)
    ax.axvline(true_p, color="k", ls="--", lw=1)
    ax.set_title(f"n={n_} (k={k_})", fontsize=10); ax.set_yticks([])
axes[0].legend(fontsize=8)
plt.suptitle("Data swamps the prior: three posteriors merge as n grows", y=1.04)
plt.tight_layout(); plt.show()


**n=10**：三个后验明显分歧（先验话语权大）；**n=2000**：三线重合——**数据淹没先验**。

**实践规则**：报告贝叶斯结果时附先验敏感性检查；若不同合理先验给出不同决策 → 老实承认"数据还不够"。
If reasonable priors disagree on the decision, the honest conclusion is "not enough data yet".


<a id="8"></a>
## 8. 实战：贝叶斯 A/B 测试 ⭐ / Bayesian A/B Testing

频率 A/B（2.8）给 p 值；贝叶斯 A/B 直接给**业务想要的量**：
- $\Pr(B > A \mid \text{data})$ —— "B 更好的概率"
- $\mathbb{E}[\text{loss}]$ —— "如果选错，期望损失多少"

做法：两组各自后验 → **蒙特卡洛采样**比较（2.12 的预演）。


In [ ]:
# A/B: 老页面 vs 新页面 / Two variants
k_A, n_A = 152, 3400        # A: 4.47%
k_B, n_B = 191, 3300        # B: 5.79%

post_A = st.beta(1 + k_A, 1 + n_A - k_A)
post_B = st.beta(1 + k_B, 1 + n_B - k_B)

# 蒙特卡洛: 从两个后验各抽 20 万个样本对比 / MC comparison
S = 200_000
sa, sb = post_A.rvs(S, random_state=1), post_B.rvs(S, random_state=2)
p_b_better = (sb > sa).mean()
lift = (sb - sa) / sa
exp_loss_choose_B = np.maximum(sa - sb, 0).mean()    # 选 B 而 A 其实更好的期望代价

print(f"P(B > A)            = {p_b_better:.1%}")
print(f"相对提升的后验中位数 = {np.median(lift):+.1%}")
print(f"95% 可信区间        = [{np.percentile(lift, 2.5):+.1%}, {np.percentile(lift, 97.5):+.1%}]")
print(f"选 B 的期望损失      = {exp_loss_choose_B:.5f} (转化率单位)")
print(f"\n频率学派对照: 两比例 z 检验 p = "
      f"{st.norm.sf(abs((k_B/n_B-k_A/n_A)/np.sqrt((k_A+k_B)/(n_A+n_B)*(1-(k_A+k_B)/(n_A+n_B))*(1/n_A+1/n_B))))*2:.4f}")

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(lift, bins=120, density=True, alpha=0.75)
ax.axvline(0, color="k", ls="--")
ax.set_xlabel("relative lift (B vs A)"); ax.set_title("Posterior of the lift — the whole story in one plot")
plt.tight_layout(); plt.show()


**贝叶斯输出的沟通优势**："B 有 99% 概率更好，提升大概在 +8%~+52% 之间，选错的期望代价几乎为零" —— 业务方**直接能用**；对比 "p=0.012, 拒绝原假设"。

⚠ 但贝叶斯 A/B **不自动免疫偷看问题**——连续监控 $\Pr(B>A)$ 并提前停同样会膨胀错误率（除非引入损失阈值决策框架）。Part 19 算总账。
Bayesian A/B is not automatically immune to peeking — continuous monitoring still inflates error rates without a proper decision framework.


<a id="9"></a>
## 9. 小结 / Summary

```
后验 ∝ 似然 × 先验
  Beta-Binomial: 更新 = 加法 (α+=k, β+=n-k)  共轭 ⭐
  先验 = 伪计数 → 冷启动 / Laplace 平滑 / 贝叶斯平均评分
  序贯: 今天后验 = 明天先验, 顺序无关
  可信区间: "95% 概率" 字面成立 (vs CI 的程序质保)
  MAP: Ridge = 高斯先验, Lasso = Laplace 先验 ⭐
  敏感性: 数据淹没先验; 先验分歧未消 = 数据不够
  贝叶斯 A/B: P(B>A) + 期望损失 — 业务能直接用的输出
```

### 💡 面试速查
1. **频率 vs 贝叶斯一句话**：θ 是常数（程序质保）vs θ 是随机变量（概率陈述）
2. **共轭**：后验同族 → 解析更新；Beta-Binomial 是标准例
3. **L2 = 高斯先验 / L1 = Laplace 先验的 MAP**——λ = σ²/τ²
4. **冷启动 = 伪计数先验**：新品 1/3 转化 ≠ 33%
5. **可信区间才支持"95% 概率在区间内"**

### 下一节
**2.11 Bootstrap & Jackknife**——第三条路：不要解析公式也不要先验，**用重抽样从数据本身榨出不确定性**。
